# Build a Transformer from Scratch - Solution Notebook

Complete implementation of a transformer language model with detailed explanations.

---

## Part 1: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 2: Dataset Preparation

In [ ]:
# Constants
SEQ_LEN = 32
VOCAB_SIZE = 1000
BATCH_SIZE = 64
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 200

class RandomSequenceDataset(Dataset):
    """Dataset that generates random token sequences."""
    
    def __init__(self, num_samples, seq_len, vocab_size):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random input and target sequences
        x = torch.randint(0, self.vocab_size, (self.seq_len,))
        y = torch.randint(0, self.vocab_size, (self.seq_len,))
        return x, y

# Create datasets
train_dataset = RandomSequenceDataset(TRAIN_SAMPLES, SEQ_LEN, VOCAB_SIZE)
val_dataset = RandomSequenceDataset(VAL_SAMPLES, SEQ_LEN, VOCAB_SIZE)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Part 3: Multi-Head Attention

**Key concepts:**
- Split d_model into num_heads of dimension d_k each
- Each head learns different aspects of relationships
- Scale attention scores by sqrt(d_k) to prevent softmax saturation
- Apply mask before softmax using -inf

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        self.rope = RotaryPositionalEmbedding(self.d_k, max_len)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project and reshape to (batch, num_heads, seq_len, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self.rope(Q, K)
        
        # Compute attention scores: Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask (set masked positions to -inf before softmax)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax to get attention weights
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Multiply by values
        attn_output = torch.matmul(attn_weights, V)
        
        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)
        output = self.W_o(attn_output)
        
        return output

## Part 4: Feed-Forward Network

**Architecture:** Linear -> GELU -> Dropout -> Linear

GELU (Gaussian Error Linear Unit) is preferred over ReLU in modern transformers.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

## Part 5: Layer Normalization

**Purpose:** Stabilizes training by normalizing activations.

Unlike BatchNorm, LayerNorm normalizes across features (last dimension) for each sample independently.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # Learnable scale and shift parameters
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Normalize along the feature dimension
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

## Part 6: Rotary Positional Embeddings (RoPE)

**Why RoPE?**
- Encodes relative positions naturally
- Better extrapolation to longer sequences
- More efficient than learned positional embeddings

**How it works:** Rotates Q and K vectors by position-dependent angles.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for RoPE"
        
        # Compute inverse frequencies for rotation
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        
        # Precompute position embeddings
        t = torch.arange(max_len, dtype=torch.float)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())
    
    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """Rotates half the hidden dims of the input."""
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = q.size(-2)
        
        # Get cached cos and sin for current sequence length
        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(0)
        
        # Apply rotation
        q_rot = q * cos + self.rotate_half(q) * sin
        k_rot = k * cos + self.rotate_half(k) * sin
        
        return q_rot, k_rot

## Part 7: Causal Mask

**Purpose:** Prevents attention to future positions during autoregressive generation.

In [ ]:
def create_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Create lower triangular causal mask."""
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

## Part 8: Transformer Decoder Block

**Pre-norm architecture:**
```
x = x + Attention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

Pre-norm is more stable than post-norm for deep models.

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        
        self.attention = MultiHeadAttention(d_model, num_heads, dropout, max_len)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Self-attention with residual connection
        attn_output = self.attention(self.norm1(x), mask)
        x = x + self.dropout1(attn_output)
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(self.norm2(x))
        x = x + self.dropout2(ff_output)
        
        return x

## Part 9: Complete Transformer Language Model

**Architecture:**
1. Token Embeddings (scaled by sqrt(d_model))
2. Stack of Transformer Blocks
3. Final Layer Norm
4. Output Projection to Vocabulary

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, max_len: int = 512,
                 dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Stack of transformer blocks
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout, max_len)
            for _ in range(num_layers)
        ])
        
        # Final layer norm and output projection
        self.norm = LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Embed tokens and scale
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        x = self.dropout(x)
        
        # Apply transformer blocks
        for layer in self.layers:
            x = layer(x, mask)
        
        # Final norm and projection
        x = self.norm(x)
        logits = self.output(x)
        
        return logits
    
    @torch.no_grad()
    def generate(self, prompt: torch.Tensor, max_new_tokens: int,
                 temperature: float = 1.0, top_k: Optional[int] = None) -> torch.Tensor:
        """Generate tokens autoregressively."""
        self.eval()
        
        for _ in range(max_new_tokens):
            # Get predictions
            mask = create_causal_mask(prompt.size(1), prompt.device)
            logits = self(prompt, mask)  # ✅ Pass mask
            
            # Get logits for last position and apply temperature
            logits = logits[:, -1, :] / temperature
            
            # Top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            prompt = torch.cat([prompt, next_token], dim=1)
        
        return prompt

## Part 10: Training Setup

In [ ]:
# Model hyperparameters
D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 4
D_FF = 256
DROPOUT = 0.1

# Initialize model
model = TransformerLM(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_len=SEQ_LEN,
    dropout=DROPOUT
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Learning rate scheduler (optional)
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

## Part 11: Training Loop

**Critical details:**
- Reshape logits to (batch×seq, vocab) and targets to (batch×seq)
- Always clip gradients to prevent exploding gradients
- Track perplexity as an interpretable metric

In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        # Move to device
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Create causal mask
        mask = create_causal_mask(batch_x.size(1), device)
        
        # Forward pass
        logits = model(batch_x, mask)
        
        # Compute loss (reshape for CrossEntropyLoss)
        loss = criterion(logits.view(-1, VOCAB_SIZE), batch_y.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        
        total_loss += loss.item()
    
    # Learning rate scheduling
    scheduler.step()
    
    # Calculate metrics
    avg_loss = total_loss / len(train_loader)
    perplexity = math.exp(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Loss: {avg_loss:.4f}")
        print(f"  Perplexity: {perplexity:.2f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")

## Part 12: Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, criterion):
    """Evaluate model on validation data."""
    model.eval()
    total_loss = 0
    
    for batch_x, batch_y in data_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        mask = create_causal_mask(batch_x.size(1), device)
        
        logits = model(batch_x, mask)
        loss = criterion(logits.view(-1, VOCAB_SIZE), batch_y.view(-1))
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(data_loader)
    perplexity = math.exp(avg_loss)
    
    return avg_loss, perplexity

# Run evaluation
val_loss, val_perplexity = evaluate(model, val_loader, criterion)
print(f"\nValidation Loss: {val_loss:.4f}")
print(f"Validation Perplexity: {val_perplexity:.2f}")

## Part 13: Text Generation

**Sampling strategies:**
- **Greedy** (temp≈0): Always pick most likely token
- **Temperature sampling**: Controls randomness
- **Top-k sampling**: Only sample from k most likely tokens

In [ ]:
# Create random prompt
prompt = torch.randint(0, VOCAB_SIZE, (1, 10)).to(device)

print("=== Generation Tests ===")

# Low temperature (more deterministic)
print("\n1. Low temperature (temp=0.1):")
generated = model.generate(prompt, max_new_tokens=20, temperature=0.1)
print(f"Generated shape: {generated.shape}")
print(f"Tokens: {generated[0].tolist()[:10]}...")  # First 10 tokens

# Normal temperature
print("\n2. Normal temperature (temp=1.0):")
generated = model.generate(prompt, max_new_tokens=20, temperature=1.0)
print(f"Generated shape: {generated.shape}")

# High temperature (more random)
print("\n3. High temperature (temp=1.5):")
generated = model.generate(prompt, max_new_tokens=20, temperature=1.5)
print(f"Generated shape: {generated.shape}")

# Top-k sampling
print("\n4. Top-k sampling (k=10, temp=1.0):")
generated = model.generate(prompt, max_new_tokens=20, temperature=1.0, top_k=10)
print(f"Generated shape: {generated.shape}")

---

## Summary

You've implemented a complete transformer language model from scratch!

**What we built:**
1. ✅ Multi-head attention mechanism
2. ✅ Rotary positional embeddings (RoPE)
3. ✅ Transformer decoder blocks with pre-norm
4. ✅ Complete language model
5. ✅ Training loop with gradient clipping
6. ✅ Evaluation and generation

**Key learnings:**
- Attention scales by sqrt(d_k) to prevent gradient issues
- Pre-norm is more stable than post-norm
- Residual connections prevent vanishing gradients
- Gradient clipping is essential for training stability
- RoPE enables better position encoding

## Next Steps

**Improvements to try:**
1. **KV Caching** - Cache keys and values for faster generation
2. **Grouped-Query Attention** - Balance efficiency and quality
3. **Flash Attention** - Use optimized attention kernels
4. **Real text data** - Train on actual text corpus
5. **Larger models** - Scale up dimensions and layers

**Debugging tips:**
- Print tensor shapes at each step
- Check for NaN values in loss
- Monitor gradient norms
- Verify mask is applied correctly
- Start small and scale up gradually